# Historical HF evaluation scratchpad

This notebook is retained as source evidence, not as an end-to-end runnable evaluation. The initial cells define a single-HF-model workflow. Later copied cells (starting with the call using `model1`, `model2`, and `model3`) are unfinished: those variables are not defined at that point and their ensemble-list argument does not match this notebook's single-model `run_test`. Do not use Run All as a reproduction recipe.

Use `eval_model.ipynb` for the three original ensemble evaluations and `../evaluation-evidence.json` for their preserved numerical results. Original code cells below are unchanged; outputs are cleared. Historical paths, models, dependencies, and data still require adaptation.


In [ ]:
import albumentations as A 
import cv2 
import numpy as np
import os
import pandas as pd 
import torch

from collections import defaultdict
from importlib import import_module
from sklearn.metrics import roc_auc_score, roc_curve
from transformers import AutoModel
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from skp.toolbox.functions import load_model_from_config

In [ ]:
class MammoDataset(Dataset):

    def __init__(self, df):
        self.df = df.copy()
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = cv2.imread(os.path.join("/mnt/stor/datasets/kaggle/rsna-breast-cancer-detection/train_cropped_png/", row.filename), 0)
        return {
            "img": img,
            "label": torch.tensor(row.cancer),
            "patient_id": row.patient_id,
            "image_id": row.image_id, 
            "laterality": row.laterality,
            "view": row["view"],
            "breast_id": row.breast_id,
            "filename": row.filename,
        }

In [ ]:
def run_test(df, model):
    pred_df = defaultdict(list)
    dataset = MammoDataset(df)
    loader = DataLoader(dataset, batch_size=1, num_workers=28, shuffle=False, drop_last=False)
    for batch in tqdm(loader, total=len(loader)):
        # reformat batch for AutoModel
        batch_img = [{"view": batch["img"][0].numpy()}]
        with torch.inference_mode():
            out = model(batch_img, device="cuda:0")
        pred_df["y_pred"].append(out["cancer"][0].item())
        pred_df["y_true"].extend(list(batch["label"].numpy()))
        pred_df["patient_id"].extend(list(batch["patient_id"].numpy()))
        pred_df["image_id"].extend(list(batch["image_id"].numpy()))
        pred_df["laterality"].extend(list(batch["laterality"]))
        pred_df["view"].extend(list(batch["view"]))
        pred_df["breast_id"].extend(list(batch["breast_id"]))
        pred_df["filename"].extend(list(batch["filename"]))
    return pd.DataFrame(pred_df)

## Split 1

In [ ]:
model = AutoModel.from_pretrained("ianpan/mammoscreen", trust_remote_code=True, token=True).eval().cuda()

In [ ]:
test_df = pd.read_csv("/mnt/stor/datasets/kaggle/rsna-breast-cancer-detection/holdout_test_v01.csv")
test_df

In [ ]:
test = run_test(test_df, model)
test

In [ ]:
roc_auc_score(test.y_true.values, test.y_pred.values)

In [ ]:
test_grouped = test.groupby("breast_id").mean(numeric_only=True)
roc_auc_score(test_grouped.y_true.values, test_grouped.y_pred.values)

In [ ]:
fpr, tpr, thresholds = roc_curve(test_grouped.y_true.values, test_grouped.y_pred.values)
# 100% sensitivity
idx = np.argmin(np.abs(1 - tpr))
print(f"Specificity: {1 - fpr[idx]:0.4f}, Sensitivity: {tpr[idx]:0.4f}, Threshold: {thresholds[idx]:0.4f}")

# 98% sensitivity
idx = np.argmin(np.abs(0.98 - tpr))
print(f"Specificity: {1 - fpr[idx]:0.4f}, Sensitivity: {tpr[idx]:0.4f}, Threshold: {thresholds[idx]:0.4f}")

# 95% sensitivity
idx = np.argmin(np.abs(0.95 - tpr))
print(f"Specificity: {1 - fpr[idx]:0.4f}, Sensitivity: {tpr[idx]:0.4f}, Threshold: {thresholds[idx]:0.4f}")

# 90% sensitivity
idx = np.argmin(np.abs(0.90 - tpr))
print(f"Specificity: {1 - fpr[idx]:0.4f}, Sensitivity: {tpr[idx]:0.4f}, Threshold: {thresholds[idx]:0.4f}")
    

In [ ]:
pred_df1 = run_test(test_df, [model1, model2, model3])

In [ ]:
pred_df1["y_pred3"] = pred_df1.y_pred0 + pred_df1.y_pred1 + pred_df1.y_pred2
for i in range(4):
    auc = roc_auc_score(pred_df1.y_true.values, pred_df1[f"y_pred{i}"].values)
    print("AUC:", f"{auc:0.4f}")

pred_df1_grouped = pred_df1.groupby("breast_id").mean(numeric_only=True)
print("\n")

for i in range(4):
    auc = roc_auc_score(pred_df1_grouped.y_true.values, pred_df1_grouped[f"y_pred{i}"].values)
    print("AUC:", f"{auc:0.4f}")

## Split 2

In [ ]:
cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
cfg.pretrained = False
cfg.load_pretrained_backbone = False
cfg.enable_gradient_checkpointing = False

weights_path = cfg.save_dir + cfg_name + "/c55fbaf7/fold0/checkpoints/ema_weights.pt"
model1 = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)

cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses_v02"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
cfg.pretrained = False
cfg.load_pretrained_backbone = False
cfg.enable_gradient_checkpointing = False

weights_path = cfg.save_dir + cfg_name + "/fa6df1c9/fold1/checkpoints/ema_weights.pt"
model2 = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)

cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses_v04"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
cfg.pretrained = False
cfg.load_pretrained_backbone = False
cfg.enable_gradient_checkpointing = False

weights_path = cfg.save_dir + cfg_name + "/cbbc1071/fold2/checkpoints/ema_weights.pt"
model3 = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)

test_df = pd.read_csv("/mnt/stor/datasets/kaggle/rsna-breast-cancer-detection/holdout_test_v02.csv")

pred_df2 = run_test(test_df, [model1, model2, model3])

In [ ]:
pred_df2["y_pred3"] = pred_df2.y_pred0 + pred_df2.y_pred1 + pred_df2.y_pred2
for i in range(4):
    auc = roc_auc_score(pred_df2.y_true.values, pred_df2[f"y_pred{i}"].values)
    print("AUC:", f"{auc:0.4f}")

pred_df2_grouped = pred_df2.groupby("breast_id").mean(numeric_only=True)
print("\n")

for i in range(4): 
    auc = roc_auc_score(pred_df2_grouped.y_true.values, pred_df2_grouped[f"y_pred{i}"].values)
    print("AUC:", f"{auc:0.4f}")

## Split 3

In [ ]:
cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
cfg.pretrained = False
cfg.load_pretrained_backbone = False
cfg.enable_gradient_checkpointing = False

weights_path = cfg.save_dir + cfg_name + "/6b00696a/fold0/checkpoints/ema_weights.pt"
model1 = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)

cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses_v02"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
cfg.pretrained = False
cfg.load_pretrained_backbone = False
cfg.enable_gradient_checkpointing = False

weights_path = cfg.save_dir + cfg_name + "/387cf1fd/fold1/checkpoints/ema_weights.pt"
model2 = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)

cfg_name = "mammo.cfg_ddsm_pretrained_aux_losses_v04"
cfg = import_module(f"skp.configs.{cfg_name}").cfg
cfg.pretrained = False
cfg.load_pretrained_backbone = False
cfg.enable_gradient_checkpointing = False

weights_path = cfg.save_dir + cfg_name + "/9c9a8feb/fold2/checkpoints/ema_weights.pt"
model3 = load_model_from_config(cfg, weights_path, device="cuda:0", eval_mode=True)

test_df = pd.read_csv("/mnt/stor/datasets/kaggle/rsna-breast-cancer-detection/holdout_test_v03.csv")

pred_df3 = run_test(test_df, [model1, model2, model3])

In [ ]:
pred_df3["y_pred3"] = pred_df3.y_pred0 + pred_df3.y_pred1 + pred_df3.y_pred2
for i in range(4):
    auc = roc_auc_score(pred_df3.y_true.values, pred_df3[f"y_pred{i}"].values)
    print("AUC:", f"{auc:0.4f}")

pred_df3_grouped = pred_df3.groupby("breast_id").mean(numeric_only=True)
print("\n")

for i in range(4): 
    auc = roc_auc_score(pred_df3_grouped.y_true.values, pred_df3_grouped[f"y_pred{i}"].values)
    print("AUC:", f"{auc:0.4f}")